**Kingdom Classification**

Dataset: https://www.kaggle.com/datasets/salikhussaini49/codon-usage

In [ ]:
import shutil

# Libraries
import time
from pathlib import Path

import joblib
import matplotlib

matplotlib.use("Agg")  # Render off-screen; figures are saved to files only
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import xgboost as xgb_lib
from sklearn.base import clone
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 99
N_JOBS = 8
CV_FOLDS = 3
DATA_PATH = Path("../data/codon_usage.csv")
RESULT_DIR = Path("../result")
RESULT_DIR.mkdir(exist_ok=True)
CACHE_DIR = Path("tuning_cache")  # Tuned models are saved here; delete cache to re-tune
CACHE_DIR.mkdir(exist_ok=True)
MODEL_NAMES = ["Random Forest", "XGBoost", "Bagging"]
MODEL_DIR = Path("../result/model")

## Data Loading and Cleaning

In [ ]:
# Obtain Dataset
df = pd.read_csv(DATA_PATH, low_memory=False)
df.head()

In [ ]:
# Dataset Information
df.info()

In [ ]:
# Numerical Statistics of Dataset
df.describe()

In [ ]:
# Check Missing Values in the Dataframe
def report_missing(data):
    missing = data.columns[data.isnull().any()]
    if missing.empty:
        print("No missing values")
    else:
        for column in missing:
            print(column, data[column].isnull().sum())


report_missing(df)

In [ ]:
# Columns "UUU" and "UUC" are read as strings because a few rows are malformed.
# Convert all codon columns to numeric and drop the rows that cannot be parsed.
codon_cols = df.columns[df.columns.get_loc("SpeciesName") + 1 :]
df[codon_cols] = df[codon_cols].apply(pd.to_numeric, errors="coerce")
print("Malformed rows dropped:", df[codon_cols].isnull().any(axis=1).sum())
df = df.dropna(subset=codon_cols).reset_index(drop=True)
report_missing(df)
df.info()

In [ ]:
# Distinct Values of The "Kingdom" Column
print(df["Kingdom"].unique())

In [ ]:
# Number of Distinct Species Names
print(df["SpeciesName"].nunique())

## Feature Engineering

In [ ]:
# Drop identifier columns, then label-encode the target ("Kingdom")
df_fix = df.drop(columns=["SpeciesID", "SpeciesName"])
encoder = LabelEncoder()
df_fix["EncodedKingdom"] = encoder.fit_transform(df_fix["Kingdom"])
df_fix = df_fix.drop(columns="Kingdom")

print("Encoded Label - Original Label:")
for i, label in enumerate(encoder.classes_):
    print(f"{i} - {label}")
report_missing(df_fix)

In [ ]:
# Split into Training (60%), Validation (20%), and Testing (20%) datasets.
# Stratified so every kingdom appears in each split.
TARGET = "EncodedKingdom"
df_train, df_val_test = train_test_split(
    df_fix, test_size=0.4, random_state=RANDOM_STATE, stratify=df_fix[TARGET]
)
df_val, df_test = train_test_split(
    df_val_test, test_size=0.5, random_state=RANDOM_STATE, stratify=df_val_test[TARGET]
)

x_train, y_train = df_train.drop(columns=TARGET), df_train[TARGET]
x_val, y_val = df_val.drop(columns=TARGET), df_val[TARGET]
x_test, y_test = df_test.drop(columns=TARGET), df_test[TARGET]

print("Number of Rows:")
print("All Data:", len(df_fix))
print("Training Data:", len(df_train))
print("Validation Data:", len(df_val))
print("Testing Data:", len(df_test))

## Helper Functions

In [ ]:
def train_model(model):
    """Fit a model on the training data and return (model, seconds taken)."""
    start = time.perf_counter()
    model.fit(x_train, y_train)
    return model, time.perf_counter() - start


def evaluate(model, name):
    """Print validation/testing reports and return testing accuracy."""
    for split, x, y in [("Validation", x_val, y_val), ("Testing", x_test, y_test)]:
        pred = model.predict(x)
        print(f"{name} - {split} Metrics")
        print(classification_report(y, pred, zero_division=0))
    return accuracy_score(y_test, model.predict(x_test))


def plot_accuracy(accuracies, title, filename):
    """Save a bar chart of testing accuracy per model."""
    fig, ax = plt.subplots()
    sns.barplot(x=MODEL_NAMES, y=accuracies, hue=MODEL_NAMES, legend=False, ax=ax)
    for i, acc in enumerate(accuracies):
        ax.text(i, acc, f"{acc:.4f}", ha="center", va="bottom")
    ax.set_title(title)
    ax.set_xlabel("Model")
    ax.set_ylabel("Accuracy")
    fig.savefig(RESULT_DIR / filename, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_training_time(times_before, times_after, filename):
    """Save a grouped bar chart of training time before/after tuning."""
    fig, ax = plt.subplots()
    width = 0.35

    positions = range(len(MODEL_NAMES))
    for offset, times, label in [
        (-width / 2, times_before, "Before Tuning"),
        (width / 2, times_after, "After Tuning"),
    ]:
        xs = [p + offset for p in positions]
        ax.bar(xs, times, width, label=label)
        for x, t in zip(xs, times):
            ax.text(x, t, f"{t:.1f}", ha="center", va="bottom")

    ax.set_xticks(list(positions))
    ax.set_xticklabels(MODEL_NAMES)
    ax.set_xlabel("Model")
    ax.set_ylabel("Time Needed to Train (in seconds)")
    ax.set_title("Comparison of Time Needed to Train The Models (in seconds)")
    ax.legend()
    ax.grid(axis="y")

    fig.savefig(RESULT_DIR / filename, dpi=150, bbox_inches="tight")
    plt.close(fig)

## Baseline Models

In [ ]:
# Train and evaluate the three baseline models
base_models = {
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
    "XGBoost": xgb_lib.XGBClassifier(
        random_state=RANDOM_STATE, tree_method="hist", n_jobs=1
    ),
    "Bagging": BaggingClassifier(
        estimator=RandomForestClassifier(random_state=RANDOM_STATE),
        random_state=RANDOM_STATE,
    ),
}

base_acc, base_time = [], []
for name, model in base_models.items():
    model, seconds = train_model(model)
    base_models[name] = model
    base_time.append(seconds)
    base_acc.append(evaluate(model, name))

plot_accuracy(
    base_acc,
    "Comparison of Accuracy Between 3 Models with Testing Data",
    "accuracy_before_tuning.png",
)

## Hyperparameter Tuning (Grid Search)

In [ ]:
# Reduced search space (4 candidates per model) with 3-fold CV to keep tuning fast
param_grids = {
    "Random Forest": {
        "n_estimators": [300],
        "max_depth": [20, 50],
        "min_samples_leaf": [1, 3],
    },
    "XGBoost": {
        "n_estimators": [300],
        "learning_rate": [0.1, 0.3],
        "max_depth": [6, 10],
    },
    "Bagging": {
        "estimator__max_depth": [8, None],
        "max_samples": [0.6, 0.75],
    },
}

tuned_models, tuned_acc, tuned_time = {}, [], []
for name in MODEL_NAMES:
    cache_file = CACHE_DIR / f"{name.replace(' ', '_')}.joblib"
    if cache_file.exists():
        cached = joblib.load(cache_file)
        model, params, seconds = cached["model"], cached["params"], cached["seconds"]
        print(f"{name}: loaded from cache")
    else:
        # Clone so workers receive a small unfitted estimator, not a fitted one
        search = GridSearchCV(
            clone(base_models[name]),
            param_grids[name],
            cv=CV_FOLDS,
            n_jobs=N_JOBS,
            verbose=2,
        )
        search, seconds = train_model(search)
        model, params = search.best_estimator_, search.best_params_
        joblib.dump({"model": model, "params": params, "seconds": seconds}, cache_file)

    print(f"{name} best parameters:", params)
    tuned_models[name] = model
    tuned_time.append(seconds)
    tuned_acc.append(evaluate(model, f"Tuned {name}"))

plot_accuracy(
    tuned_acc,
    "Comparison of Accuracy Between 3 Models with Testing Data After Tuning",
    "accuracy_after_tuning.png",
)

## Training Time Comparison

In [ ]:
# "After Tuning" time is the full grid search duration (all candidates, 3-fold CV)
plot_training_time(base_time, tuned_time, "training_time.png")
print("Saved figures:", sorted(p.name for p in RESULT_DIR.glob("*.png")))

## Save Best Model

In [ ]:
# Pick the single best model out of all six (baseline + tuned) by validation
# accuracy, so the testing set stays untouched for the final report.
MODEL_DIR.mkdir(exist_ok=True)

candidates = {}
for name in MODEL_NAMES:
    candidates[(name, "baseline")] = base_models[name]
    candidates[(name, "tuned")] = tuned_models[name]

val_scores = {
    key: accuracy_score(y_val, m.predict(x_val)) for key, m in candidates.items()
}

best_key = max(val_scores, key=val_scores.get)
best_algorithm, best_version = best_key
best_model = candidates[best_key]

print(f"Best: {best_algorithm} ({best_version})")
print(f"Validation accuracy: {val_scores[best_key]:.4f}")
print(f"Testing accuracy:    {accuracy_score(y_test, best_model.predict(x_test)):.4f}")

# Saved together with the feature order and label names needed for inference
model_path = MODEL_DIR / f"model_{best_algorithm.lower().replace(' ', '_')}.joblib"
joblib.dump(
    {
        "model": best_model,
        "features": list(x_train.columns),
        "kingdoms": list(encoder.classes_),
    },
    model_path,
)

print("Saved:", model_path)

## Inference

In [ ]:
def predict_kingdom(model_file, samples):
    """Predict kingdom labels from a DataFrame of codon-usage features.

    `samples` must contain the same columns as the training features
    (DNAtype, Ncodons and the 64 codon frequencies); order does not matter.
    """
    bundle = joblib.load(model_file)
    missing = set(bundle["features"]) - set(samples.columns)
    if missing:
        raise ValueError(f"Missing feature columns: {sorted(missing)}")
    codes = bundle["model"].predict(samples[bundle["features"]])
    return [bundle["kingdoms"][int(c)] for c in codes]


# Example: classify 5 rows from the testing set
sample = x_test.head(5)
print("Predicted:", predict_kingdom(model_path, sample))
print("Actual:   ", [encoder.classes_[i] for i in y_test.head(5)])

In [ ]:
# Everything is saved at this point, so the tuning cache is no longer needed and can be deleted
shutil.rmtree(CACHE_DIR, ignore_errors=True)